# 6b — Double Dissociation Ablation

Extends notebook 6: ablate concept-only neurons and measure log P(keyword) on
**both** concept prompts and checker prompts.

Expected: concept-only ablation degrades concept prompts, has zero/minimal effect
on checker prompts. Shared/random ablation impacts both.

**GPU required (A100 recommended).**

In [ ]:
# Cell 1 – Dependencies & configuration
import subprocess, sys, os

pkgs = ["h5py", "transformers", "torch", "numpy", "pandas", "tqdm",
        "pyarrow", "accelerate", "scipy"]
for pkg in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

LANG = "P"
MODEL = "QW"
PREFIX = f"{LANG}_{MODEL}_"
MODEL_CONFIGS = {
    "QW": {"id": "Qwen/Qwen2.5-Coder-7B",                "n_layers": 28, "mlp_dim": 3584},
    "DS": {"id": "deepseek-ai/deepseek-coder-6.7b-base",  "n_layers": 32, "mlp_dim": 4096},
}
MODEL_ID = MODEL_CONFIGS[MODEL]["id"]
N_LAYERS = MODEL_CONFIGS[MODEL]["n_layers"]
MLP_DIM = MODEL_CONFIGS[MODEL]["mlp_dim"]

N_PROMPTS = None  # None = use ALL available prompts
BATCH_SIZE = 8
LAYERS = list(range(13, 20))  # layers 13-19 only (peak region)
SEED = 42

# Target concepts: same 10 as notebook 6 for Python
if LANG == "P":
    CONCEPTS = {
        "ast__Import": "import", "ast__Assert": "assert", "ast__Break": "break",
        "ast__For": "for", "ast__While": "while", "ast__Try": "try",
        "ast__If": "if", "ast__FunctionDef": "def", "ast__Lambda": "lambda",
        "builtin__len": "len",
    }
else:  # Rust top 10 by concept fraction
    CONCEPTS = {
        "rust__Crate": "crate", "rust__Super": "super", "rust__Match": "match",
        "rust__Break": "break", "rust__Self": "self", "rust__Loop": "loop",
        "rust__While": "while", "rust__For": "for", "rust__Return": "return",
        "rust__If": "if",
    }

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import shutil
    mp = "/content/drive"
    subprocess.run(["fusermount", "-uz", mp], capture_output=True)
    if os.path.isdir(mp):
        shutil.rmtree(mp, ignore_errors=True)
    drive.mount(mp)
    DATA_DIR = "/content/drive/MyDrive/DATA/New-Atlas"
else:
    DATA_DIR = "/Users/piotrwilam/Data/New-Atlas"

LOCAL_SRC = "/Users/piotrwilam/Code/New-Atlas/src"
COLAB_SRC = "/content/drive/MyDrive/CODE/New-Atlas/src"
SRC_PATH = LOCAL_SRC if os.path.isdir(LOCAL_SRC) else COLAB_SRC
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

NEURON_CSV = f"{PREFIX}4_neuron_list_eps0.5_cons0.8"
OBJECT_PARQUET = f"{PREFIX}1A_object_prompts.parquet"
CHECKER_PARQUET = f"{PREFIX}1B_checker_prompts.parquet"

os.makedirs(DATA_DIR, exist_ok=True)
print(f"Cell: {LANG}_{MODEL} | Concepts: {len(CONCEPTS)}")

In [ ]:
# Cell 2 – Imports, model loading, hooks
import ast as ast_mod
import numpy as np
import pandas as pd
import torch
import json
import re
import time
import logging
from tqdm.auto import tqdm
from scipy import stats
from transformers import AutoTokenizer, AutoModelForCausalLM

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("6b")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto",
).eval()

print(f"Model loaded | Device: {DEVICE}")

In [ ]:
# Cell 3 – Tokenizer verification
target_token_ids = {}
for concept, keyword in CONCEPTS.items():
    ids = tokenizer.encode(keyword, add_special_tokens=False)
    target_token_ids[concept] = ids[0]
    print(f"  {concept:25s} keyword='{keyword}' -> id={ids[0]}")

In [ ]:
# Cell 4 – Load neuron lists (eps=0.5 AND eps=0.1, plus union)

def load_neuron_list(data_dir, prefix, eps, cons, obj_type="both"):
    import os
    p1 = os.path.join(data_dir, f"{prefix}4_neuron_list_eps{eps}_cons{cons}_layers_part1_{obj_type}.csv")
    p2 = os.path.join(data_dir, f"{prefix}4_neuron_list_eps{eps}_cons{cons}_layers_part2_{obj_type}.csv")
    if os.path.exists(p1) and os.path.exists(p2):
        return pd.concat([pd.read_csv(p1), pd.read_csv(p2)], ignore_index=True)
    old = os.path.join(data_dir, f"{prefix}4_neuron_list_eps{eps}_cons{cons}_all_layers_{obj_type}.csv")
    if os.path.exists(old):
        return pd.read_csv(old)
    return None

# Load both threshold settings
df_05 = load_neuron_list(DATA_DIR, PREFIX, "0.5", "0.8")
df_01 = load_neuron_list(DATA_DIR, PREFIX, "0.1", "0.8")

print(f"Neuron list eps=0.5: {len(df_05) if df_05 is not None else 'MISSING'} rows")
print(f"Neuron list eps=0.1: {len(df_01) if df_01 is not None else 'MISSING'} rows")

# Build neuron data with all three sets per concept per layer
neuron_data = {}  # (concept, layer) -> {"co_05": [...], "co_01": [...], "co_union": [...], "both_05": [...], "token_only_05": [...]}

for concept in CONCEPTS:
    for lid in LAYERS:
        entry = {"co_05": [], "co_01": [], "co_union": [], "both_05": [], "token_only_05": []}
        
        if df_05 is not None:
            row = df_05[(df_05["object"] == concept) & (df_05["layer"] == lid)]
            if len(row) > 0:
                entry["co_05"] = ast_mod.literal_eval(row.iloc[0]["concept_only"])
                entry["both_05"] = ast_mod.literal_eval(row.iloc[0]["both"])
                entry["token_only_05"] = ast_mod.literal_eval(row.iloc[0]["token_only"])
        
        if df_01 is not None:
            row = df_01[(df_01["object"] == concept) & (df_01["layer"] == lid)]
            if len(row) > 0:
                entry["co_01"] = ast_mod.literal_eval(row.iloc[0]["concept_only"])
        
        # Union
        entry["co_union"] = sorted(set(entry["co_05"]) | set(entry["co_01"]))
        
        neuron_data[(concept, lid)] = entry

# Summary
print("\nNeuron counts per concept (layer 14):")
for concept in CONCEPTS:
    nd = neuron_data.get((concept, 14), {})
    n05 = len(nd.get("co_05", []))
    n01 = len(nd.get("co_01", []))
    nunion = len(nd.get("co_union", []))
    short = concept.split("__")[1]
    print(f"  {short:15s} eps0.5={n05:4d} eps0.1={n01:4d} union={nunion:4d}")


In [ ]:
# Cell 5 – Load BOTH prompt types

rng = np.random.default_rng(SEED)

# Object (concept) prompts
df_obj = pd.read_parquet(os.path.join(DATA_DIR, OBJECT_PARQUET))
obj_col = "ast_node" if LANG == "P" else "construct"

concept_prompts = {}  # concept -> list of prompt strings
for concept, keyword in CONCEPTS.items():
    if LANG == "P":
        prefix_type, name = concept.split("__", 1)
        if prefix_type == "ast":
            pool = df_obj[df_obj[obj_col] == name]
        else:
            pool = df_obj[df_obj["builtin_obj"] == name]
    else:
        name = concept.split("__", 1)[1]
        pool = df_obj[df_obj[obj_col] == name]

    sampled = pool.sample(frac=1, random_state=SEED) if N_PROMPTS is None else pool.sample(n=min(N_PROMPTS, len(pool)), random_state=SEED)
    concept_prompts[concept] = sampled["prompt_text"].tolist()

# Checker prompts
df_chk = pd.read_parquet(os.path.join(DATA_DIR, CHECKER_PARQUET))

checker_prompts = {}  # concept -> list of prompt strings
for concept, keyword in CONCEPTS.items():
    chk_rows = df_chk[df_chk["object"] == concept]
    if len(chk_rows) == 0:
        checker_prompts[concept] = []
        continue
    sampled = chk_rows.sample(frac=1, random_state=SEED) if N_PROMPTS is None else chk_rows.sample(n=min(N_PROMPTS, len(chk_rows)), random_state=SEED)
    checker_prompts[concept] = sampled["prompt_text"].tolist()

for concept in CONCEPTS:
    print(f"  {concept:25s} obj={len(concept_prompts[concept]):3d} chk={len(checker_prompts[concept]):3d}")

del df_obj, df_chk

In [ ]:
# Cell 6 – AblationHook + log-prob utilities (from notebook 6)

class AblationHook:
    def __init__(self, model, layer_id, neuron_indices, mode="zero"):
        self.neuron_indices = neuron_indices
        self.handle = None
        module_dict = dict(model.named_modules())
        mlp_name = f"model.layers.{layer_id}.mlp"
        if mlp_name not in module_dict:
            raise ValueError(f"MLP module not found: {mlp_name}")
        self.module = module_dict[mlp_name]

    def _hook_fn(self, module, input, output):
        if isinstance(output, tuple):
            out = output[0]
        else:
            out = output
        modified = out.clone()
        modified[:, :, self.neuron_indices] = 0.0
        if isinstance(output, tuple):
            return (modified,) + output[1:]
        return modified

    def register(self):
        self.handle = self.module.register_forward_hook(self._hook_fn)
        return self

    def remove(self):
        if self.handle is not None:
            self.handle.remove()
            self.handle = None


def find_target_positions(input_ids, target_token_id):
    positions = []
    for b in range(input_ids.shape[0]):
        ids = input_ids[b].tolist()
        found = None
        for i, tid in enumerate(ids):
            if tid == target_token_id and i > 0:
                found = i
                break
        positions.append(found)
    return positions


def get_logprobs_batch(model, tokenizer, prompts, target_token_id, device):
    encoded = tokenizer(
        prompts, return_tensors="pt", padding=True,
        truncation=True, add_special_tokens=False,
    ).to(device)
    positions = find_target_positions(encoded["input_ids"], target_token_id)
    with torch.no_grad():
        outputs = model(input_ids=encoded["input_ids"], attention_mask=encoded["attention_mask"])
        logits = outputs.logits
    results = []
    for b, pos in enumerate(positions):
        if pos is None:
            results.append((None, None))
        else:
            log_probs = torch.log_softmax(logits[b, pos - 1, :], dim=-1)
            lp = log_probs[target_token_id].item()
            results.append((lp, pos))
    return results


def sample_shared_neurons(both_indices, n_needed, rng):
    if len(both_indices) <= n_needed:
        return list(both_indices)
    return rng.choice(both_indices, size=n_needed, replace=False).tolist()


def sample_random_neurons(concept_only, both, token_only, n_needed, mlp_dim, rng):
    excluded = set(concept_only) | set(both) | set(token_only)
    available = [i for i in range(mlp_dim) if i not in excluded]
    if len(available) <= n_needed:
        return available
    return rng.choice(available, size=n_needed, replace=False).tolist()


print("Ablation utilities ready.")

In [ ]:
# Cell 7 – Main double dissociation loop (5 conditions, layers 13-19)

results = []
ablation_rng = np.random.default_rng(SEED)
start_time = time.time()

for concept_idx, (concept, keyword) in enumerate(CONCEPTS.items()):
    obj_prompts = concept_prompts[concept]
    chk_prompts = checker_prompts[concept]
    target_tid = target_token_ids[concept]

    if len(obj_prompts) == 0 or len(chk_prompts) == 0:
        print(f"  SKIP {concept}: obj={len(obj_prompts)} chk={len(chk_prompts)}")
        continue

    print(f"\n[{concept_idx+1}/{len(CONCEPTS)}] {concept} (obj={len(obj_prompts)} chk={len(chk_prompts)})")

    for layer_id in tqdm(LAYERS, desc="Layers", leave=False):
        nd = neuron_data.get((concept, layer_id))
        if nd is None:
            continue

        co_05 = nd["co_05"]
        co_01 = nd["co_01"]
        co_union = nd["co_union"]
        both_05 = nd["both_05"]
        token_only_05 = nd["token_only_05"]

        if len(co_union) == 0:
            continue

        # Random: same size as union
        random_sample = sample_random_neurons(co_union, both_05, token_only_05,
                                               len(co_union), MLP_DIM, ablation_rng)

        conditions = {}
        if len(co_05) > 0:
            conditions["co_eps05"] = co_05
        if len(co_01) > 0:
            conditions["co_eps01"] = co_01
        if len(co_union) > 0:
            conditions["co_union"] = co_union
        if len(random_sample) > 0:
            conditions["random"] = random_sample
        # Shared: same size as union from both_05
        shared_sample = sample_shared_neurons(both_05, len(co_union), ablation_rng)
        if len(shared_sample) > 0:
            conditions["shared"] = shared_sample

        # Clean forward passes on BOTH prompt types
        def get_clean_logps(prompts):
            clean_data = []
            for batch_start in range(0, len(prompts), BATCH_SIZE):
                batch = prompts[batch_start:batch_start + BATCH_SIZE]
                batch_results = get_logprobs_batch(model, tokenizer, batch, target_tid, DEVICE)
                for i, (lp, pos) in enumerate(batch_results):
                    if lp is not None:
                        clean_data.append((batch_start + i, lp, pos))
            return clean_data

        clean_obj = get_clean_logps(obj_prompts)
        clean_chk = get_clean_logps(chk_prompts)

        if len(clean_obj) == 0 and len(clean_chk) == 0:
            continue

        # Ablated forward passes
        for cond_name, cond_indices in conditions.items():
            if len(cond_indices) == 0:
                continue

            hook = AblationHook(model, layer_id, cond_indices)
            hook.register()

            def get_ablated_logps(prompts):
                ablated = {}
                for batch_start in range(0, len(prompts), BATCH_SIZE):
                    batch = prompts[batch_start:batch_start + BATCH_SIZE]
                    batch_results = get_logprobs_batch(model, tokenizer, batch, target_tid, DEVICE)
                    for i, (lp, pos) in enumerate(batch_results):
                        if lp is not None:
                            ablated[batch_start + i] = lp
                return ablated

            abl_obj = get_ablated_logps(obj_prompts)
            abl_chk = get_ablated_logps(chk_prompts)

            hook.remove()

            deltas_obj = [clean_lp - abl_obj[pidx]
                          for pidx, clean_lp, _ in clean_obj if pidx in abl_obj]
            deltas_chk = [clean_lp - abl_chk[pidx]
                          for pidx, clean_lp, _ in clean_chk if pidx in abl_chk]

            results.append({
                "concept": concept,
                "keyword": keyword,
                "layer": layer_id,
                "condition": cond_name,
                "n_ablated": len(cond_indices),
                "n_obj_prompts": len(deltas_obj),
                "n_chk_prompts": len(deltas_chk),
                "mean_delta_obj": float(np.mean(deltas_obj)) if deltas_obj else np.nan,
                "mean_delta_chk": float(np.mean(deltas_chk)) if deltas_chk else np.nan,
                "std_delta_obj": float(np.std(deltas_obj)) if deltas_obj else np.nan,
                "std_delta_chk": float(np.std(deltas_chk)) if deltas_chk else np.nan,
            })

    elapsed = time.time() - start_time
    print(f"  {elapsed/60:.1f} min elapsed")

print(f"\nDone: {len(results)} rows in {(time.time()-start_time)/60:.1f} min")


In [ ]:
# Cell 8 – Analyze double dissociation
import matplotlib.pyplot as plt

df_results = pd.DataFrame(results)

# For each concept at each layer: is there a dissociation?
# concept_only ablation: high delta_obj, low delta_chk
# shared ablation: delta_obj ≈ delta_chk
# random ablation: low both

print("DOUBLE DISSOCIATION SUMMARY")
print("=" * 70)

for concept in CONCEPTS:
    co = df_results[(df_results["concept"] == concept) & (df_results["condition"] == "co_union")]
    if len(co) == 0:
        continue

    # Average across layers
    mean_obj = co["mean_delta_obj"].mean()
    mean_chk = co["mean_delta_chk"].mean()
    ratio = mean_obj / mean_chk if abs(mean_chk) > 0.001 else float('inf')

    dissoc = "YES" if mean_obj > 0.1 and abs(mean_chk) < 0.05 else "partial" if mean_obj > mean_chk * 2 else "no"
    short = concept.split("__")[1]
    print(f"  {short:15s} delta_obj={mean_obj:.4f} delta_chk={mean_chk:.4f} ratio={ratio:.1f} dissociation={dissoc}")

# Plot: concept-only ablation effect on obj vs chk prompts
co_all = df_results[df_results["condition"] == "co_union"].copy()
if len(co_all) > 0:
    fig, ax = plt.subplots(figsize=(8, 8))
    for concept in CONCEPTS:
        sub = co_all[co_all["concept"] == concept]
        if len(sub) == 0:
            continue
        ax.scatter(sub["mean_delta_obj"], sub["mean_delta_chk"],
                   alpha=0.5, s=15, label=concept.split("__")[1])

    lim = max(abs(co_all["mean_delta_obj"]).max(), abs(co_all["mean_delta_chk"]).max()) * 1.1
    ax.plot([-lim, lim], [-lim, lim], "k--", alpha=0.3, label="y=x")
    ax.axhline(0, color="gray", linestyle="-", alpha=0.3)
    ax.axvline(0, color="gray", linestyle="-", alpha=0.3)
    ax.set_xlabel("Delta log P on concept prompts")
    ax.set_ylabel("Delta log P on checker prompts")
    ax.set_title(f"{LANG}_{MODEL}: Double Dissociation (concept-only ablation)")
    ax.legend(fontsize=7, ncol=2)
    plt.tight_layout()
    plt.savefig(os.path.join(DATA_DIR, f"{PREFIX}6b_double_dissociation_scatter.png"), dpi=150)
    plt.show()

In [ ]:
# Cell 9 – Condition comparison heatmap

# For each condition: mean delta on obj vs chk, averaged across layers
summary_rows = []
for concept in CONCEPTS:
    for cond in ["concept_only", "shared", "random"]:
        sub = df_results[(df_results["concept"] == concept) & (df_results["condition"] == cond)]
        if len(sub) == 0:
            continue
        summary_rows.append({
            "concept": concept.split("__")[1],
            "condition": cond,
            "delta_obj": sub["mean_delta_obj"].mean(),
            "delta_chk": sub["mean_delta_chk"].mean(),
        })

df_summary = pd.DataFrame(summary_rows)

if len(df_summary) > 0:
    print("\nCondition comparison (mean across layers):")
    pivot_obj = df_summary.pivot(index="concept", columns="condition", values="delta_obj")
    pivot_chk = df_summary.pivot(index="concept", columns="condition", values="delta_chk")

    print("\n--- Delta on CONCEPT prompts ---")
    display(pivot_obj.reindex(columns=[c for c in ["co_eps05", "co_eps01", "co_union", "shared", "random"] if c in pivot_obj.columns]))

    print("\n--- Delta on CHECKER prompts ---")
    display(pivot_chk.reindex(columns=[c for c in ["co_eps05", "co_eps01", "co_union", "shared", "random"] if c in pivot_chk.columns]))

In [ ]:
# Cell 10 – Save results
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)

out_path = os.path.join(DATA_DIR, f"{PREFIX}6b_double_dissociation.csv")
df_results.to_csv(out_path, index=False)
print(f"Saved: {out_path}")

summary_path = os.path.join(DATA_DIR, f"{PREFIX}6b_double_dissociation_summary.csv")
df_summary.to_csv(summary_path, index=False)
print(f"Saved: {summary_path}")

print(f"\n6b complete.")

try:
    from google.colab import runtime
    runtime.unassign()
except (ImportError, AttributeError):
    print("Not running on Colab -- skipping unassign.")